# 01 – OCR extraction

Runs Tesseract OCR on every labelled PDF page and stores the **words** and their **normalised bounding boxes** (0–1000 scale, as LayoutLM expects). The result is the input for `02_layoutlm_classification.ipynb`.

**Input:** label CSVs with one row per page (`path`, `page_num`, `label`, ...) pointing to the PDFs.
**Output:** `data/pages_ocr.json` (JSON lines, one row per page, with `words` and `bbox` columns).

Requires the Tesseract binary (`tesseract-ocr`) to be installed on your system, see the README.

# Setup

In [ ]:
# ---- Configuration: edit these paths for your setup --------------------------
LABELS_DIR = "data/labels"      # folder with the label CSVs
OUTPUT_PATH = "data/pages_ocr.json"

# The label CSVs store the absolute path of each PDF as it was on the original machine.
# If your PDFs live somewhere else, set the old prefix and the new one; paths get rewritten on load.
OLD_PATH_PREFIX = "/content/drive/MyDrive/Mining documents/"
NEW_PATH_PREFIX = "/path/to/your/pdfs/"   # set to OLD_PATH_PREFIX if nothing changed

In [ ]:
import os

import numpy as np
import pandas as pd
import pymupdf
import pytesseract
from PIL import Image
from tqdm import tqdm

# Read labelled pages

In [ ]:
def load_labels(csv_name, drop_labels=("table",)):
    """Load a label CSV, drop unwanted labels and drop every page that appears twice
    (same path + page_num), since a duplicated page has an ambiguous label."""
    df = pd.read_csv(os.path.join(LABELS_DIR, csv_name))
    df = df[~df["label"].isin(drop_labels)]
    duplicates = df[df.duplicated(subset=["path", "page_num"], keep=False)]
    df = df.drop(duplicates.index).reset_index(drop=True)
    df["path"] = df["path"].str.replace(OLD_PATH_PREFIX, NEW_PATH_PREFIX, regex=False)
    return df


train_data = load_labels("YESAB_data_train.csv")
valid_data = load_labels("YESAB_data_test.csv")

merged_data = pd.concat([train_data, valid_data], ignore_index=True)
merged_data["source"] = "YESAB"
merged_data.shape, merged_data["label"].value_counts()

# Get text and bounding boxes

In [ ]:
def normalize_box(box, width, height):
    """Scale a pixel box (x0, y0, x1, y1) to the 0-1000 range used by LayoutLM."""
    return [
        int(1000 * (box[0] / width)),
        int(1000 * (box[1] / height)),
        int(1000 * (box[2] / width)),
        int(1000 * (box[3] / height)),
    ]


def extract_text_and_bboxes(example):
    """Render one PDF page to an image, run Tesseract and add `words` and `bbox` to the row."""
    document = pymupdf.open(example["path"])
    page = document.load_page(example["page_num"] - 1)
    pix = page.get_pixmap()
    image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    width, height = image.size

    ocr_df = pytesseract.image_to_data(image, output_type="data.frame")
    float_cols = ocr_df.select_dtypes("float").columns
    ocr_df = ocr_df.dropna().reset_index(drop=True)
    ocr_df[float_cols] = ocr_df[float_cols].round(0).astype(int)
    # drop empty / whitespace-only detections
    ocr_df = ocr_df.replace(r"^\s*$", np.nan, regex=True)
    ocr_df = ocr_df.dropna().reset_index(drop=True)

    words = [str(w) for w in ocr_df.text]
    boxes = []
    for _, row in ocr_df[["left", "top", "width", "height"]].iterrows():
        x, y, w, h = tuple(row)
        boxes.append(normalize_box([x, y, x + w, y + h], width, height))
    assert len(words) == len(boxes)

    example["words"] = words
    example["bbox"] = boxes
    return example

## Run OCR on the YESAB pages

This is slow (a few seconds per page), so the result is saved straight away.

In [ ]:
tqdm.pandas()
merged_data_OCR = merged_data.progress_apply(extract_text_and_bboxes, axis=1)
merged_data_OCR.to_json("data/yesab_ocr.json", orient="records", lines=True)

# Add the correspondence pages

In [ ]:
correspondences = pd.read_csv(os.path.join(LABELS_DIR, "correspondences_data.csv"))
correspondences["path"] = correspondences["path"].str.replace(OLD_PATH_PREFIX, NEW_PATH_PREFIX, regex=False)
correspondences["source"] = "Correspondences"

correspondences_OCR = correspondences.progress_apply(extract_text_and_bboxes, axis=1)
correspondences_OCR.to_json("data/correspondences_ocr.json", orient="records", lines=True)

# Merge and save the final dataset

In [ ]:
merged_data_with_correspondences_OCR = pd.concat([merged_data_OCR, correspondences_OCR], ignore_index=True)
merged_data_with_correspondences_OCR.to_json(OUTPUT_PATH, orient="records", lines=True)
merged_data_with_correspondences_OCR.shape, merged_data_with_correspondences_OCR["label"].value_counts()